In [ ]:
# =========================================================
# IMPORTS
# =========================================================
import pandas as pd
import numpy as np
import os

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from scipy.sparse import hstack, csr_matrix

# =========================================================
# LOAD DATA
# =========================================================
def load_data(filepath="ml_recipes.csv"):
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"[ERROR] File '{filepath}' not found!")

    df = pd.read_csv(filepath)

    required_cols = ["name", "ingredients", "cuisine", "time"]
    df = df[required_cols].copy()

    df.dropna(inplace=True)

    df["name"] = df["name"].str.lower().str.strip()
    df["ingredients"] = df["ingredients"].str.lower().str.strip()
    df["cuisine"] = df["cuisine"].str.lower().str.strip()
    df["time"] = pd.to_numeric(df["time"], errors="coerce")

    df.dropna(inplace=True)
    df.reset_index(drop=True, inplace=True)

    print(f"[INFO] Loaded {len(df)} recipes")
    return df

# =========================================================
# FEATURE ENGINEERING
# =========================================================
def create_features(df):
    tfidf = TfidfVectorizer(max_features=300)
    ing_matrix = tfidf.fit_transform(df["ingredients"])

    ohe = OneHotEncoder(handle_unknown="ignore")
    cuisine_matrix = ohe.fit_transform(df[["cuisine"]])

    scaler = MinMaxScaler()
    time_scaled = scaler.fit_transform(df[["time"]])
    time_sparse = csr_matrix(time_scaled)

    features = hstack([ing_matrix, cuisine_matrix, time_sparse])

    return features, tfidf, ohe, scaler

# =========================================================
# SAVE RESULTS
# =========================================================
def save_results(results, filename="recommendations.csv"):
    df = pd.DataFrame(results)
    df.to_csv(filename, index=False)
    print(f"[INFO] Results saved to {filename}")

# =========================================================
# RECOMMEND BY NAME
# =========================================================
def recommend_by_name(name, df, cosine_sim, euclidean_dist, top_n=5):
    name = name.lower().strip()

    if name not in df["name"].values:
        print(f"[ERROR] '{name}' not found in dataset")
        return []

    idx = df[df["name"] == name].index[0]

    cos_scores = list(enumerate(cosine_sim[idx]))
    euc_scores = list(enumerate(euclidean_dist[idx]))

    cos_scores = sorted(cos_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]
    euc_scores = sorted(euc_scores, key=lambda x: x[1])[1:top_n+1]

    results = []

    print(f"\n=== Recommendations for: {name.title()} ===\n")

    print("Cosine Similarity:")
    for i, score in cos_scores:
        recipe = df.iloc[i]
        print(f"{recipe['name']} | {recipe['cuisine']} | {recipe['time']} min | Score: {score:.4f}")
        results.append({"method": "cosine", "name": recipe["name"], "cuisine": recipe["cuisine"], "time": recipe["time"], "score": score})

    print("\nEuclidean Distance:")
    for i, score in euc_scores:
        recipe = df.iloc[i]
        print(f"{recipe['name']} | {recipe['cuisine']} | {recipe['time']} min | Distance: {score:.4f}")
        results.append({"method": "euclidean", "name": recipe["name"], "cuisine": recipe["cuisine"], "time": recipe["time"], "score": score})

    return results

# =========================================================
# CUSTOM RECOMMENDATION (USER INPUT)
# =========================================================
def recommend_custom(ingredients, cuisine, time, df, features, tfidf, ohe, scaler, top_n=5):
    ing_vec = tfidf.transform([ingredients.lower()])
    cuisine_vec = ohe.transform([[cuisine.lower()]])
    time_vec = scaler.transform([[float(time)]])
    time_sparse = csr_matrix(time_vec)

    query = hstack([ing_vec, cuisine_vec, time_sparse])

    scores = cosine_similarity(query, features).flatten()
    top_indices = scores.argsort()[::-1][:top_n]

    results = []

    print("\n=== Custom Recommendations ===\n")
    for i in top_indices:
        recipe = df.iloc[i]
        print(f"{recipe['name']} | {recipe['cuisine']} | {recipe['time']} min | Score: {scores[i]:.4f}")
        results.append({"name": recipe["name"], "cuisine": recipe["cuisine"], "time": recipe["time"], "score": scores[i]})

    return results

# =========================================================
# MAIN MENU SYSTEM
# =========================================================
def main():
    df = load_data()

    features, tfidf, ohe, scaler = create_features(df)

    cosine_sim = cosine_similarity(features)
    euclidean_dist = euclidean_distances(features)

    print("\n====== 🍽️ Recipe Recommendation System ======\n")

    while True:
        print("\nChoose Option:")
        print("1. Recommend by Recipe Name")
        print("2. Recommend by Ingredients (Custom)")
        print("3. Exit")

        choice = input("Enter choice (1/2/3): ")

        if choice == "1":
            name = input("Enter recipe name: ")
            results = recommend_by_name(name, df, cosine_sim, euclidean_dist)
            save_results(results, "name_based.csv")

        elif choice == "2":
            ingredients = input("Enter ingredients: ")
            cuisine = input("Enter cuisine type: ")
            time = input("Enter time (minutes): ")

            try:
                time = float(time)
            except:
                print("[ERROR] Invalid time input")
                continue

            results = recommend_custom(
                ingredients, cuisine, time,
                df, features, tfidf, ohe, scaler
            )

            save_results(results, "custom_based.csv")

        elif choice == "3":
            print("👋 Exiting...")
            break

        else:
            print("[ERROR] Invalid choice")

# =========================================================
# RUN
# =========================================================
if __name__ == "__main__":
    main()